# 进阶实验：LlamaIndex RAG 改进与扩展

## 实验目标
- 学会调整文档切分策略（chunk size / overlap）；
- 更换 embedding 模型，观察检索效果变化；
- 自定义 Prompt，提高回答的可控性；
- 尝试使用不同的 Top-K 值进行检索。

> 提示：本 Notebook 假设你已经完成了基础实验，并理解了 RAG 的基本流程。


In [10]:
# 若尚未安装，请先安装依赖：
%pip install llama-index sentence-transformers transformers faiss-cpu huggingface-hub llama-index-embeddings-huggingface
%pip install llama-index-llms-ollama

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple

   -------------------- ------------------- 1/2 [llama-index-llms-ollama]
   ---------------------------------------- 2/2 [llama-index-llms-ollama]

Note: you may need to restart the kernel to use updated packages.


In [2]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# 1. 读取文档
documents = SimpleDirectoryReader("./docs").load_data()
print(f"共加载到 {len(documents)} 个文档。")

d:\miniconda\envs\py310\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
d:\miniconda\envs\py310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


共加载到 2 个文档。


## TODO 1：调整文档切分策略

请阅读下面的代码，并尝试：
- 修改 `chunk_size`（如 128 / 256 / 512）；
- 修改 `chunk_overlap`（如 10 / 50）；
- 观察切分后的节点数量变化。

In [4]:
# --- TODO 1：调整 SentenceSplitter 的参数并进行对比实验 ---
from llama_index.core.node_parser import SentenceSplitter

configs = [
    {"size": 128, "overlap": 20},
    {"size": 256, "overlap": 20},
    {"size": 512, "overlap": 50}
]

print("=== TODO 1: 文本切分策略对比实验 ===")

for config in configs:
    splitter = SentenceSplitter(
        chunk_size=config["size"], 
        chunk_overlap=config["overlap"]
    )
    test_nodes = splitter.get_nodes_from_documents(documents)
    
    print(f"\n[配置] Chunk Size: {config['size']}, Chunk Overlap: {config['overlap']}")
    print(f" -> 切分后得到的节点总数: {len(test_nodes)}")
    
    if test_nodes:
        first_node_text = test_nodes[0].get_content()
        # 修复反斜杠报错：先处理换行符，再进行切片和打印
        clean_text = first_node_text.replace('\n', ' ')
        summary = clean_text[:60]
        print(f" -> 首个节点文本长度: {len(first_node_text)}")
        print(f" -> 首个节点内容摘要: {summary}...")

# 最终选择一组参数用于后续实验
splitter = SentenceSplitter(chunk_size=256, chunk_overlap=30)
nodes = splitter.get_nodes_from_documents(documents)
print("-" * 40)
print(f"最终选择参数配置 (256/30)，共得到 {len(nodes)} 个节点用于后续索引构建。")

=== TODO 1: 文本切分策略对比实验 ===

[配置] Chunk Size: 128, Chunk Overlap: 20
 -> 切分后得到的节点总数: 9
 -> 首个节点文本长度: 75
 -> 首个节点内容摘要: 什么是 RAG ？  RAG（Retrieval-Augmented Generation，检索增强生成）是一种结合“信...

[配置] Chunk Size: 256, Chunk Overlap: 20
 -> 切分后得到的节点总数: 3
 -> 首个节点文本长度: 243
 -> 首个节点内容摘要: 什么是 RAG ？  RAG（Retrieval-Augmented Generation，检索增强生成）是一种结合“信...

[配置] Chunk Size: 512, Chunk Overlap: 50
 -> 切分后得到的节点总数: 2
 -> 首个节点文本长度: 390
 -> 首个节点内容摘要: 什么是 RAG ？  RAG（Retrieval-Augmented Generation，检索增强生成）是一种结合“信...
----------------------------------------
最终选择参数配置 (256/30)，共得到 3 个节点用于后续索引构建。


## TODO 2：更换 Embedding 模型

下面示例使用 `sentence-transformers/all-MiniLM-L6-v2` 模型，
可以尝试更换成其他中文或多语言模型（如 BAAI/bge-base-zh）。

In [6]:
# --- TODO 2：替换 / 调整 embedding 模型名称并对比检索效果 ---
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, Settings

# 定义我们要对比的模型（假设你本地有这些路径，或者使用 HF 上的名称）
model_paths = {
    "MiniLM (轻量级)": "../vector-database-demo/all-MiniLM-L6-v2",
    "BGE (高性能)": "BAAI/bge-small-zh-v1.5" # 如果你有联网或本地有 BGE 模型可以开启此行对比
}

test_question = "RAG 的主要优点有哪些？"

print("=== TODO 2: Embedding 模型检索效果对比 ===")

for name, path in model_paths.items():
    print(f"\n正在加载模型: {name}...")
    try:
        # 1. 加载模型
        curr_embed_model = HuggingFaceEmbedding(model_name=path)
        
        # 2. 临时构建一个索引进行测试
        # 注意：为了公平对比，这里使用 TODO 1 中切分好的 nodes
        temp_index = VectorStoreIndex(nodes, embed_model=curr_embed_model)
        retriever = temp_index.as_retriever(similarity_top_k=2)
        
        # 3. 执行检索
        retrieval_results = retriever.retrieve(test_question)
        
        print(f"[{name}] 针对问题 '{test_question}' 检索到的最相关片段：")
        for i, res in enumerate(retrieval_results):
            # 处理换行符防止显示混乱
            content = res.node.get_content().replace('\n', ' ')
            print(f"  Rank {i+1} (Score: {res.score:.4f}): {content[:80]}...")
            
    except Exception as e:
        print(f"加载模型 {name} 失败: {e}")

# --- 最终确定生产环境使用的模型 ---
# 这里保持你原来的路径
embed_model = HuggingFaceEmbedding(model_name="../vector-database-demo/all-MiniLM-L6-v2")
Settings.embed_model = embed_model
print("\n[设置] 已将全局默认 Embedding 设置为 all-MiniLM-L6-v2")

2026-01-23 17:31:55,435 - INFO - Load pretrained SentenceTransformer: ../vector-database-demo/all-MiniLM-L6-v2


=== TODO 2: Embedding 模型检索效果对比 ===

正在加载模型: MiniLM (轻量级)...


2026-01-23 17:31:55,652 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-zh-v1.5


[MiniLM (轻量级)] 针对问题 'RAG 的主要优点有哪些？' 检索到的最相关片段：
  Rank 1 (Score: 0.3573): 将每个 chunk 编码成向量，并存入向量数据库； 4. 用户提问时，将问题编码成向量，在向量库中做相似度检索，取出 top-k 个相关文本； 5. 把检索到的...
  Rank 2 (Score: 0.3322): 什么是 RAG ？  RAG（Retrieval-Augmented Generation，检索增强生成）是一种结合“信息检索”和“文本生成”的技术。 它的核心...

正在加载模型: BGE (高性能)...


2026-01-23 17:32:02,004 - INFO - 1 prompt is loaded, with the key: query
2026-01-23 17:32:02,094 - INFO - Load pretrained SentenceTransformer: ../vector-database-demo/all-MiniLM-L6-v2


[BGE (高性能)] 针对问题 'RAG 的主要优点有哪些？' 检索到的最相关片段：
  Rank 1 (Score: 0.5648): 什么是 RAG ？  RAG（Retrieval-Augmented Generation，检索增强生成）是一种结合“信息检索”和“文本生成”的技术。 它的核心...
  Rank 2 (Score: 0.3951): 将每个 chunk 编码成向量，并存入向量数据库； 4. 用户提问时，将问题编码成向量，在向量库中做相似度检索，取出 top-k 个相关文本； 5. 把检索到的...

[设置] 已将全局默认 Embedding 设置为 all-MiniLM-L6-v2


## TODO 3：基于节点构建索引，并设置 Top-K

请完成以下代码：
- 基于 `nodes` 构建 `VectorStoreIndex`；
- 在 `as_query_engine` 中传入 `similarity_top_k` 参数；
- 尝试不同的 top_k 值（如 1 / 3 / 5），观察回答差异。

In [ ]:
# 确保先执行：pip install llama-index-llms-ollama
from llama_index.llms.ollama import Ollama
from llama_index.core import VectorStoreIndex, Settings

# 配置大模型 - 确保 Ollama 服务已在后台启动
Settings.llm = Ollama(model="qwen3:0.6b", base_url="http://127.0.0.1:11434", request_timeout=120.0)

# --- TODO 3：构建索引与查询引擎对比实验 ---
# 1. 基于 TODO 1 中生成的 nodes 构建索引
index = VectorStoreIndex(nodes)

top_k_list = [1, 3, 5]
question = "RAG 的主要优点有哪些？"

print(f"=== TODO 3: Similarity Top-K 对比实验 ===")
print(f"测试问题: {question}\n")

for k in top_k_list:
    print(f"正在测试 Top-K = {k} ...")
    
    # 2. 构建查询引擎，传入当前循环的 k 值
    query_engine = index.as_query_engine(similarity_top_k=k)
    
    # 3. 执行查询
    response = query_engine.query(question)
    
    # 4. 格式化输出
    print(f"【Top-K 为 {k} 时的回答】：")
    print(response)
    print(f"依据的原始片段数量: {len(response.source_nodes)}")
    print("-" * 50)

# 最终选择一个配置用于后续（如 TODO 4）
query_engine = index.as_query_engine(similarity_top_k=3)

=== TODO 3: Similarity Top-K 对比实验 ===
测试问题: RAG 的主要优点有哪些？

正在测试 Top-K = 1 ...


2026-01-23 19:37:31,392 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/show "HTTP/1.1 200 OK"
2026-01-23 19:37:53,504 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


【Top-K 为 1 时的回答】：
RAG的主要优点包括：  
1. **控制知识来源**，避免模型“瞎编”；  
2. **方便更新知识库**，无需重新训练大模型；  
3. **适合企业/课程/项目文档问答系统**。
依据的原始片段数量: 1
--------------------------------------------------
正在测试 Top-K = 3 ...


2026-01-23 19:38:07,771 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


【Top-K 为 3 时的回答】：
RAG的主要优点包括：  
1. **显式控制知识来源**，避免模型“瞎编”；  
2. **方便更新知识库**，无需重新训练大模型；  
3. **适合基于企业/课程/项目文档的问答系统**。
依据的原始片段数量: 3
--------------------------------------------------
正在测试 Top-K = 5 ...


2026-01-23 19:38:19,458 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


【Top-K 为 5 时的回答】：
RAG的主要优点包括：  
1. 显式控制知识来源，避免模型“瞎编”；  
2. 方便更新知识库，无需重新训练大模型；  
3. 适合基于企业/课程/项目文档的问答系统。
依据的原始片段数量: 3
--------------------------------------------------


: 

## TODO 4：自定义 Prompt（可选进阶）

你可以尝试为查询引擎设计一个更“严格”的 Prompt，例如：
- 必须标明回答依据的文档（或片段）；
- 如果文档中没有相关信息，必须明确说“无法回答”。

可以参考 LlamaIndex 文档中关于自定义 Query Engine / Chat Engine 的部分，
在此基础上扩展本实验。
